#### LCEL 문법 개요

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model= 'llama3.2:1b')

# from langchain_anthropic import ChatAnthropic

# llm = ChatAnthropic(model= 'claude-3-haiku-20240307')

In [36]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser   # 이게 이번 장의 주제

# 1. 프롬프트의 템플릿을 만들고
chat_prompt_template = ChatPromptTemplate.from_messages([
  ('system', '당신은 국제 금융과 경영에 해박한 전문가입니다.'),
  ('human', '{subject}의 개념과 예시에 대해서 설명하시오')
])

# 2. 템플릿의 placeholder에 값을 입력해 프롬프트를 완성한 뒤
chat_prompt = chat_prompt_template.invoke({'subject': "대손충당금"})

# 3. LLM에 프롬프트를 먹여 output을 뽑아서
output = llm.invoke(chat_prompt)

# 4. outputParser에 넣는다
output_parser = StrOutputParser()   # Str으로 Output을 가져오겠다~

answer = output_parser.invoke(output)

# answer = 
# output_parser.invoke(llm.invoke(chat_prompt_template.invoke({'subject': "대손충당금"})))

In [29]:
# output_parser.invoke(llm.invoke(chat_prompt_template.invoke({'subject': "대손충당금"})))
# 이런 형식으로 작성하기엔 너무 복잡하기 때문에 고안된 것이 LCEL 문법

chain = chat_prompt_template | llm | output_parser    ## 이 chain도 runnable 이므로 invoke()로 호출할 수 있음
chain_answer = chain.invoke({'subject': "대손충당금"})

In [30]:
answer

'대손충당금은 기업이 판매한 상품이나 제공한 용역에 대해 고객이 지불하지 않을 것으로 예상되는 금액을 미리 계상하는 것입니다. 이를 통해 기업은 회계 보수주의 원칙에 따라 실제 발생할 수 있는 손실을 반영할 수 있습니다.\n\n예를 들어, 기업이 1억원의 매출채권을 보유하고 있다고 가정합시다. 과거 경험상 이 중 2%가 회수되지 않을 것으로 예상된다면, 기업은 2,000,000원의 대손충당금을 설정해야 합니다. 이를 통해 매출채권의 실제 회수가능액 98,000,000원이 재무상태표에 반영됩니다.\n\n대손충당금은 대손상각비라는 비용 계정을 통해 손익계산서에 반영되며, 이는 기업의 영업이익을 감소시킵니다. 또한 대손충당금은 매출채권에 대한 차감계정으로 재무상태표에 표시됩니다.\n\n이러한 대손충당금 설정은 기업의 재무제표가 보다 현실적으로 표현되도록 하며, 잠재적인 손실을 사전에 반영함으로써 재무건전성을 제고하는 데 기여합니다.'

In [31]:
chain_answer

'대손충당금은 회계상 미수금 등의 채권에 대해 미래에 발생할 수 있는 대손 손실을 예상하여 미리 비용으로 인식하는 것입니다. 이를 통해 재무제표 상 채권의 실제 회수가능가액을 반영할 수 있습니다.\n\n대손충당금의 예시는 다음과 같습니다:\n\n1. 거래처의 부도 위험이 높은 미수금에 대해 과거 경험률을 반영하여 10%의 대손충당금을 설정한 경우\n2. 연체기간이 6개월 이상인 미수금에 대해 50%의 대손충당금을 설정한 경우 \n3. 소송이 진행 중인 채권에 대해 80%의 대손충당금을 설정한 경우\n\n이처럼 과거 경험, 채권의 연체기간, 법적 소송 여부 등을 고려하여 합리적으로 대손충당금을 설정해야 합니다. 이를 통해 재무제표 상 채권의 실제 회수가능가액을 더욱 정확하게 반영할 수 있습니다.'

#### 활용 예시

#### 1. ChatPromptTemplate - context 유지

In [32]:
# 1. 프롬프트 템플릿 작성 by ChatPromptTemplate - 메시지 구조(Chat)로 전달하는 프롬프트 템플릿
# - 다중 메시지 구조 (맥락 관리 가능, 대화형 에이전트)
ceo_prompt_chat = ChatPromptTemplate.from_messages([
  ('system', '당신은 국제 기업들에 대해서 해박한 전문가입니다.'),
  ('human', '기업 {company_name}의 CEO의 이름만 제시해줘')
])


# 2. chain 생성
ceo_chain_chat = ceo_prompt_chat | llm | output_parser

# 3. 생성된 chain을 통한 질의
ceo_samsung_chat = ceo_chain_chat.invoke({'company_name': 'Samsung'})


print(ceo_samsung_chat)


Samsung의 현재 CEO는 한종희 회장입니다.


#### 2. PromptTemplate - 단순한 질의응답

In [33]:
from langchain_core.prompts import PromptTemplate


# 1. 프롬프트 템플릿 작성 by PromptTemplate - 단일한 template와 input_variables로 구성되는 프롬프트
# - 단순한 텍스트나 문자열 형태의 질의응답에 적합
ceo_prompt = PromptTemplate(
  template= '''
  기업 {company}의 CEO가 누구인지 그 이름만 제시해줘
  ''',
  input_variables=["company"]
)


# 2. chain 생성
ceo_chain = ceo_prompt | llm | output_parser

# 3. 생성된 chain을 통한 질의
ceo_samsung = ceo_chain.invoke({'company': 'Samsung'})


print(ceo_samsung)


Samsung의 CEO는 이재용입니다.


#### 3. runnablepassthrough

In [34]:
# 원래 invoke()에는 key와 value들의 쌍을 딕셔너리의 형태로 전달하여야 한다.

## runnablepassthrough - 프롬프트 내의 key가 하나일 경우, 단일한 value만 전달해도 되는 방법
## placeholder가 하나인, 단순한 케이스의 경우에 추천됨


from langchain_core.runnables import RunnablePassthrough

quick_chain = {'company': RunnablePassthrough()} | ceo_chain | llm |output_parser

quick_samsung = quick_chain.invoke('Samsung')

print(quick_samsung)

네, 맞습니다. 삼성전자의 CEO(최고경영자)는 이재용 부회장입니다.

이재용 부회장은 삼성 창업주 이병철 회장의 손자이자 이건희 회장의 아들로, 2014년부터 삼성전자의 실질적인 경영을 맡고 있습니다. 

삼성전자는 세계적인 전자 기업으로, 스마트폰, 반도체, TV 등 다양한 분야에서 시장을 선도하고 있습니다. 이재용 부회장의 경영 능력과 전략이 삼성전자의 성장에 많은 기여를 해왔다고 평가받고 있습니다.


#### 4. Chain 자체도 (runnable이므로) 다른 Chain의 일부가 될 수 있다.

In [ ]:
'''
앞서 만들어본 체인 자체도 runnable이기 때문에
chain을 구성요소로 하는 chain 또한 만들어 볼 수 있다.
''' 


ceo_descrive_chain = quick_chain | chat_prompt_template | llm | output_parser

CEO_descrive = ceo_descrive_chain.invoke('모건 스탠리')    ## ceo 맞추는 chain -> 개념 설명 chain -> LLM -> parser

print(CEO_descrive)


### 앞선 체인의 결과물이 단순한 단어로 이어지기 때문에
### context 전달이 잘 되고 있지 않는 모습

네, 알겠습니다. 금융 경영 분야에서 리더십의 개념과 주요 예시에 대해 설명드리겠습니다.

리더십(Leadership)이란 조직의 목표를 달성하기 위해 구성원들을 효과적으로 이끌고 동기부여하는 능력을 말합니다. 금융 및 경영 분야에서 뛰어난 리더십을 발휘한 대표적인 사례로는 다음과 같은 예시를 들 수 있습니다.

1. 제프 베즈os (Amazon 창업자 및 전 CEO)
- 아마존을 세계 최대의 온라인 유통업체로 성장시킨 리더십
- 끊임없는 혁신과 장기적 관점의 경영으로 회사를 혁신
- 고객 중심의 경영 철학으로 시장을 선도

2. 스티브 잡스 (Apple 공동 창업자)
- Apple의 부흥을 이끌어낸 카리스마 넘치는 리더십
- 혁신적인 제품 개발과 강력한 브랜드 이미지 구축
- 창의성과 디자인 혁신을 중시하는 경영 방식

3. 제이미 다이먼 (JP모건 체이스 CEO)
- 2008년 금융위기 속에서도 회사를 안정적으로 이끌어냄
- 위험관리와 자본 확충에 주력하며 건전성 제고
- 주주와 고객, 직원들의 이해관계를 균형있게 고려

이처럼 금융 및 경영 분야에서 성공적인 리더십을 발휘한 경영자들은 공통적으로 혁신, 고객 중심의 경영, 위기관리 능력 등이 두드러졌습니다. 이를 통해 조직의 성장과 발전을 이끌어 내었다고 볼 수 있습니다.


#### 계정별 유의점과 감사 절차 가이드

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

concern_prompt = ChatPromptTemplate.from_messages([
  ('system', '기말 감사에 나선 인차지 회계사의 입장에서 신입 회계사가 해당 계정과목과 관련해 주의할 점을 설명해주세요'),
  ('human', '신입 회계사가 {item} 계정과목과 관련해서 기말 감사 절차에서 주의할 점과 감사 절차에 대해서 알려줘')
])
## 이 부분 HumanMessage 프롬프트에서는 감사 절차와 유의사항을 한번에 물어본 점이 문제이다
## 생성형 LLM과 같은 경우는 프롬프트를 최대한 쪼개는 것이 좋다.

newstep_chain = {'item': RunnablePassthrough()} | concern_prompt | llm | output_parser

advice = newstep_chain.invoke('현금 및 현금성자산')

In [ ]:
print(advice)

# claude까지 쓴 결과는 만족스럽긴 한데,
# 나중에 감사기준서나 회계기준 RAG 파이프라인으로 구현해보고 싶다.

현금 및 현금성자산 계정은 기업의 재무상태를 나타내는 중요한 계정과목입니다. 신입 회계사가 현금 및 현금성자산 계정에 대한 기말 감사 시 주의해야 할 점은 다음과 같습니다.

1. 현금시재 실사
- 기말 시점에 현금시재에 대한 실사를 수행하여 실재성을 확인해야 합니다.
- 현금시재 실사 시 관찰하고 실사내역을 기록하여 증거를 확보해야 합니다.

2. 은행계좌 조회
- 은행 계좌 잔액 내역을 직접 확인하여 회계기록과 일치하는지 확인해야 합니다.
- 은행 계좌 잔액 확인 시 은행 발행 잔액증명서를 입수하여 증거로 확보해야 합니다.

3. 현금성자산 검토
- 현금성자산(단기 투자자산, 단기금융상품 등)의 실재성과 평가의 적정성을 검토해야 합니다.
- 현금성자산의 만기, 유동성, 안전성 등을 고려하여 현금성자산 분류의 적절성을 판단해야 합니다.

4. 내부통제 검토
- 현금 및 현금성자산 관리를 위한 내부통제 시스템의 설계와 운영 효과성을 평가해야 합니다.
- 현금 수불 내역, 승인 절차, 보안관리 등 내부통제 활동을 검토해야 합니다.

5. 거래 내역 검토
- 현금 및 현금성자산의 증감 거래 내역을 샘플링하여 적정성을 검토해야 합니다.
- 거래 승인, 분개, 계정 분류 등이 회계기준에 부합하는지 확인해야 합니다.

이처럼 현금 및 현금성자산 계정에 대한 실사, 확인, 분석, 내부통제 검토 등 다양한 감사 절차를 수행하여 회계기록의 정확성과 재무제표의 공정성을 확보해야 합니다.


#### 보론) Safety 절차에 대해서

In [ ]:
'''
LLM을 이용해 LangChain을 구축 할 때 Safety 절차를 구현하는 것은 꽤나 중요하다.

외부에서 Prompt 인젝션을 통해 나 또는 클라이언트의 중요 정보가 유출될 수도 있기 때문이다.


이때 safety 절차는 프롬프트에 반영되는 것이 아니라,

Chain의 일부 중, 해당 질문이 정보보안 우려가 있는지 검증하고 (Y/N)
Y라면 Full logic으로,
N라면 생성을 중지하는 식의 safety chain으로 구현하는 것이다.

이렇게 함으로써 컴퓨팅 자원도 절약할 수 있고, safety chain도 무거운 모델을 쓸 필요가 없다는 장점이 있다.

추가적으로 그러한 시도를 서버에 저장하는 방법도 생각해 볼 수 있겠다!!!
'''